# 06 — Uplift Analysis and Targeting

**Business question:** Which customers are likely to change their churn behaviour *because* they receive the offer?

This differs from churn prediction:
- churn model → who is likely to leave?
- uplift model → whose behaviour is likely to change because of treatment?

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from src.causal import t_learner_uplift
from src.decision import expected_customer_value, optimize_threshold

path = ROOT/"data/processed/demo_retention_experiment.csv"
if not path.exists():
    exec((ROOT/"data/generate_demo_data.py").read_text())
    main()
df = pd.read_csv(path)

In [ ]:
features = [
    "tenure","monthly_charges","total_charges","contract_type",
    "payment_method","tech_support","internet_service","num_services"
]
Xraw = df[features]
num = Xraw.select_dtypes(include="number").columns
cat = Xraw.select_dtypes(exclude="number").columns

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median")), num),
    ("cat", make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ), cat)
])
X = prep.fit_transform(Xraw)

uplift, model_c, model_t = t_learner_uplift(
    X, df["treatment"].values, df["churn_after_campaign"].values
)
df["predicted_churn_reduction"] = uplift
df[["customer_id","predicted_churn_reduction"]].sort_values(
    "predicted_churn_reduction", ascending=False
).head(10)

In [ ]:
df["customer_value"] = df["expected_monthly_margin"] * df["expected_remaining_months"]
df["treatment_cost_if_targeted"] = 30.0

df["expected_net_value"] = expected_customer_value(
    df["predicted_churn_reduction"],
    df["expected_monthly_margin"],
    df["expected_remaining_months"],
    df["treatment_cost_if_targeted"]
)

best, threshold_table = optimize_threshold(
    df["predicted_churn_reduction"],
    df["customer_value"],
    df["treatment_cost_if_targeted"]
)
best

In [ ]:
target = df[df["predicted_churn_reduction"] >= best[0]].copy()
target[[
    "customer_id","predicted_churn_reduction","expected_net_value"
]].sort_values("expected_net_value", ascending=False).head(20)

## Methodological caution

The demo treatment data are simulated. Replace them with a real randomized experiment
(e.g. Criteo Uplift) before making empirical causal claims.

The portfolio value here is demonstrating that you understand the difference between:
risk prediction, average treatment effect, heterogeneous treatment effect, and financial targeting.